# **ETL Excel to PostgreSQL using Docker**

In [96]:
# Import library yang diperlukan

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date # untuk mendefine production date, membuat label data yang kita extract
from sqlalchemy import create_engine

In [97]:
# Load data dari file Excel

df = pd.read_excel(r"E:\Self Learning\03. Learn Python\Python_Workspace\ETL_Excel_with_Pandas\Shift Report Tipe A Februari 2026.xlsx", 
                   sheet_name="01", 
                   skiprows=9,
                   usecols="A:O",
                   nrows=25)

In [98]:
# Drop kolom yang tidak diperlukan

df.drop(df.columns[[1,3,4,5]],axis=1, inplace=True) 
# axis untuk menjelaskan kita ada di kolom axis=1
# inplace=True untuk menyimpan perubahan di df, kalau False maka perubahan tidak disimpan di df

df.columns

Index(['Unnamed: 0', 'Unnamed: 2', 'Product', 'D.Time', '122T...', 'Product.1',
       'D.Time.1', '122T....1', 'Product.2', 'D.Time.2', '122T....2'],
      dtype='object')

In [99]:
# Rename kolom agar lebih mudah dipahami

df.rename(columns={"Unnamed: 0" : "Section",
                   "Unnamed: 2" : "Product",
                   "Product" : "Shift_2",
                   "D.Time" : "Downtime_shift_2",
                   "122T..." : "Tangki_shift_2",
                   "Product.1" : "Shift_3",
                   "D.Time.1" : "Downtime_shift_3",
                   "122T....1" : "Tangki_shift_3",
                   "Product.2" : "Shift_1",
                   "D.Time.2" : "Downtime_shift_1",
                   "122T....2" : "Tangki_shift_1"}, inplace=True)

In [100]:
# Mengubah nama kolom menjadi huruf kecil semua agar lebih konsisten

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ", "_") # optional untuk mengganti spasi dengan underscore agar lebih mudah diakses
df.columns = df.columns.str.strip() # optional untuk menghapus spasi di awal dan akhir nama kolom (Seperti trim di SQL)
df.head()

,section,product,shift_2,downtime_shift_2,tangki_shift_2,shift_3,downtime_shift_3,tangki_shift_3,shift_1,downtime_shift_1,tangki_shift_1
0,101,Feed Stream (C16-18 Raw Ester),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Oleic Intermediate Stream,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Heavy Phase Fraction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,102,Feed Stream (Hydrogenated C16-18),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Feed Stream (Hydrogenated Process Stock),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [101]:
# Mengisi nilai yang kosong (NaN)

df['section'] = df['section'].ffill() # untuk mengisi nilai yang kosong dengan nilai sebelumnya (forward fill)
df.head()

,section,product,shift_2,downtime_shift_2,tangki_shift_2,shift_3,downtime_shift_3,tangki_shift_3,shift_1,downtime_shift_1,tangki_shift_1
0,101,Feed Stream (C16-18 Raw Ester),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,101,Oleic Intermediate Stream,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,101,Heavy Phase Fraction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,102,Feed Stream (Hydrogenated C16-18),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,102,Feed Stream (Hydrogenated Process Stock),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [102]:
# Membuat kolom baru untuk mengkategorikan jenis aktivitas berdasarkan nama produk

df['activity'] = np.where(df['product'].str.contains("Feed", case=False, na=False), "Feed", "Production")
df['production_dates'] = pd.to_datetime("2026-02-01")
df.head()

,section,product,shift_2,downtime_shift_2,tangki_shift_2,shift_3,downtime_shift_3,tangki_shift_3,shift_1,downtime_shift_1,tangki_shift_1,activity,production_dates
0,101,Feed Stream (C16-18 Raw Ester),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Feed,2026-02-01
1,101,Oleic Intermediate Stream,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Production,2026-02-01
2,101,Heavy Phase Fraction,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Production,2026-02-01
3,102,Feed Stream (Hydrogenated C16-18),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Feed,2026-02-01
4,102,Feed Stream (Hydrogenated Process Stock),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Feed,2026-02-01


In [103]:
# Melakukan melt pada kolom shift untuk mengubah data dari format wide menjadi long agar lebih mudah dianalisis dan dimasukkan ke database

df_melted_prod = df.melt(id_vars=['section', 'activity', 'production_dates'],
                    value_vars=['shift_1', 'shift_2', 'shift_3'],
                    var_name='shift_category',
                    value_name='shift_value')

df_melted_prod['shift_value'] = df_melted_prod['shift_value'].fillna(0)
df_melted_prod['shift_category'] = df_melted_prod['shift_category'].str.replace("shift_", "Shift ")

df_melted_dt = df.melt(id_vars=['section', 'activity', 'production_dates'],
                    value_vars=['downtime_shift_1', 'downtime_shift_2', 'downtime_shift_3'],
                    var_name='shift_category',
                    value_name='downtime_value')

df_melted_dt['downtime_value'] = df_melted_dt['downtime_value'].fillna(0)
df_melted_dt['shift_category'] = df_melted_dt['shift_category'].str.replace("downtime_shift_", "Shift ")

df_melted_tangki = df.melt(id_vars=['section', 'activity', 'production_dates'],
                    value_vars=['tangki_shift_1', 'tangki_shift_2', 'tangki_shift_3'],
                    var_name='shift_category',
                    value_name='tangki_value')

df_melted_tangki['tangki_value'] = df_melted_tangki['tangki_value'].fillna("-")
df_melted_tangki['shift_category'] = df_melted_tangki['shift_category'].str.replace("tangki_shift_", "Shift ")

df_melted_tangki


,section,activity,production_dates,shift_category,tangki_value
0,101,Feed,2026-02-01,Shift 1,-
1,101,Production,2026-02-01,Shift 1,-
2,101,Production,2026-02-01,Shift 1,-
3,102,Feed,2026-02-01,Shift 1,-
4,102,Feed,2026-02-01,Shift 1,-
...,...,...,...,...,...
64,104,Feed,2026-02-01,Shift 3,-
65,104,Feed,2026-02-01,Shift 3,-
66,104,Feed,2026-02-01,Shift 3,-
67,104,Production,2026-02-01,Shift 3,T129


In [104]:
# untuk menggabungkan df_melted_prod dan df_melted_dt berdasarkan kolom yang sama (section, activity, production_dates, shift_category) 
df_finalize = df_melted_prod.merge(df_melted_dt, on=['section', 'activity', 'production_dates', 'shift_category']) 

# untuk menggabungkan df_finalize dan df_melted_tangki berdasarkan kolom yang sama (section, activity, production_dates, shift_category)
df_finalize = df_finalize.merge(df_melted_tangki, on=['section', 'activity', 'production_dates', 'shift_category'])

df_finalize

,section,activity,production_dates,shift_category,shift_value,downtime_value,tangki_value
0,101,Feed,2026-02-01,Shift 1,0.0,0.0,-
1,101,Production,2026-02-01,Shift 1,0.0,0.0,-
2,101,Production,2026-02-01,Shift 1,0.0,0.0,-
3,101,Production,2026-02-01,Shift 1,0.0,0.0,-
4,101,Production,2026-02-01,Shift 1,0.0,0.0,-
...,...,...,...,...,...,...,...
1180,104,Production,2026-02-01,Shift 3,0.0,0.0,-
1181,104,Production,2026-02-01,Shift 3,0.0,0.0,T129
1182,104,Production,2026-02-01,Shift 3,0.0,0.0,-
1183,104,Production,2026-02-01,Shift 3,0.0,0.0,T129


In [ ]:
# Membuat container PostgreSQL menggunakan Docker
# docker run -d --name postgres-db -e POSTGRES_USER=admin -e POSTGRES_PASSWORD=admin123 -e POSTGRES_DB=mydatabase -p 5433:5432 postgres


**docker run -d --name postgres-db -e POSTGRES_USER=admin -e POSTGRES_PASSWORD=admin123 -e POSTGRES_DB=mydatabase -p 5433:5432 postgres**

Penjelasan Docker
* docker --> membangunkan engine docker
* run --> menjalankan container yang ada, jika container nya tidak ada, docker akan mencari di komunitas docker
* -d --> untuk running secara background
* --name --> menamakan nama container kita
* postgres-db --> nama container kita
* -e POSTGRES_USER --> environment nama User PostgreSQL nya
* -e POSTGRES_PASSWORD --> environment password PostgreSQL nya
* -e POSTGRES_DB --> environment nama database PostgreSQL nya
* -p 5433:5432 --> port 5433 port untuk physical machine (Host) dan port 5432 adalah port untuk container nya. Kenapa menggunakan 5433? Karena sudah terdapat PostgreSQL diluar Container
* postgres --> nama library nya di docker community

Kemudian script diatas di running di terminal

In [ ]:
########################################################################################
# WARNING: Pastikan value di bawah ini tidak boleh di publish ke publik atau di upload ke repository, 
# karena ini adalah informasi sensitif yang bisa digunakan untuk mengakses database kita. 
# Untuk project yang lebih besar, sebaiknya gunakan environment variables atau file konfigurasi yang tidak di upload ke repository untuk menyimpan informasi ini.
########################################################################################

DB_USER = "admin"
DB_PASS = "admin123"
DB_HOST = "localhost"
DB_PORT = "5433" # default port untuk PostgreSQL adalah 5432, tapi karena sudah ada postgreSQL existing yang menggunakan port 5432, kita mapping ke 5433 di host 
DB_NAME = "mydatabase"

In [111]:
# Membuat engine untuk koneksi ke database PostgreSQL menggunakan SQLAlchemy

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [113]:
# Menyimpan data ke dalam tabel di database PostgreSQL menggunakan method to_sql dari pandas

df_finalize.to_sql(name="report_type_a", con=engine, if_exists="replace", index=False)

185